# 🚗 QUY TRÌNH TIỀN XỬ LÝ DỮ LIỆU BIỂN SỐ XE Ô TÔ (VNLP DATASET)

Notebook này thực hiện toàn bộ quy trình tiền xử lý, làm sạch và chuẩn hóa dữ liệu phát hiện biển số xe ô tô từ thư mục giải nén `C:\Users\lenovo\Downloads\VNLP_detection` sang thư mục dự án `data/VNLP_detection`.

### Các bước xử lý chính trong Pipeline:
1. **Loại bỏ xe máy:** Chỉ quét 2 thư mục ô tô `one_row` (biển dài) và `two_rows` (biển vuông), loại bỏ hoàn toàn `two_rows_label_xe_may`.
2. **Làm sạch mã biển số (Plate ID):** Chuẩn hóa chuỗi biển số và cắt bỏ tiền tố gõ nhầm (`BC`, `BD`, `B`).
3. **Phát hiện & Loại bỏ 25 ảnh chụp dọc dị biệt ($H > W$):** Lọc bỏ 25 bức ảnh chụp dọc ($480 \times 640$) bị lỗi lệch nhãn do tác giả bộ dữ liệu gốc hoán đổi trục tọa độ.
4. **Chống rò rỉ dữ liệu (Anti-Data Leakage):** Gom nhóm toàn bộ ảnh đầu xe và đuôi xe của cùng một chiếc xe theo `Plate ID`. Chia tập Train (80%) / Val (10%) / Test (10%) theo danh tính từng chiếc xe.
5. **Chuẩn hóa nhãn sang format YOLO:** Đọc kích thước thực tế (W, H) của từng ảnh, chuyển đổi tọa độ (X1, Y1, X2, Y2) sang (x_center, y_center, width, height) chuẩn hóa [0, 1] và sinh các file `.txt` tương ứng.
6. **Tự động sinh file cấu hình `data.yaml`** sẵn sàng để huấn luyện mô hình YOLO.

In [21]:
# 1. KHAI BÁO THƯ VIỆN VÀ TỰ ĐỘNG NHẬN DIỆN ĐƯỜNG DẪN DỰ ÁN
import os
import re
import time
import random
import shutil
from pathlib import Path
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

# Tự động lấy thư mục gốc của project (nơi chứa file notebook này)
PROJECT_ROOT = Path.cwd()

# 1. Thư mục đích lưu dữ liệu chuẩn hóa (nằm ngay trong project, độc lập mọi máy tính)
DST_ROOT = PROJECT_ROOT / "data" / "VNLP_detection"

# 2. Tự động tìm thư mục dữ liệu nguồn
candidate_paths = [
    PROJECT_ROOT / "data" / "raw" / "detection",
    PROJECT_ROOT / "data" / "raw" / "VNLP_detection" / "detection",
    Path.home() / "Downloads" / "VNLP_detection" / "detection",
    Path.home() / "Downloads" / "detection"
]

SRC_ROOT = None
for p in candidate_paths:
    if p.exists() and (p / "one_row").exists():
        SRC_ROOT = p
        break

if SRC_ROOT is None:
    raise FileNotFoundError("Không tìm thấy thư mục ảnh gốc. Hãy đảm bảo folder detection tồn tại trong Downloads hoặc data/raw/!")

print(f"Thư mục dự án  : {PROJECT_ROOT}")
print(f"Thư mục nguồn  : {SRC_ROOT} (Tự động nhận diện OK)")
print(f"Thư mục đích   : {DST_ROOT} (Độc lập mọi máy tính)")

Thư mục dự án  : c:\Users\lenovo\licenseplate-detection
Thư mục nguồn  : C:\Users\lenovo\Downloads\VNLP_detection\detection (Tự động nhận diện OK)
Thư mục đích   : c:\Users\lenovo\licenseplate-detection\data\VNLP_detection (Độc lập mọi máy tính)


## 2. Quét danh sách ảnh ô tô và Làm sạch mã biển số (Plate ID)
* Chỉ lấy ảnh từ `one_row` và `two_rows`.
* Bỏ qua thư mục xe máy.
* Cắt bỏ các tiền tố gõ nhầm (`BC`, `BD`, `B`) trong chuỗi biển số.

In [22]:
def clean_plate_id(plate_raw: str) -> str:
    """Chuẩn hóa chuỗi biển số và cắt bỏ tiền tố gõ nhầm (BC, BD, B)."""
    plate = plate_raw.upper()
    m = re.match(r"^(BC|BD|B)(\d{2}[A-Z].*)", plate)
    if m:
        return m.group(2)
    return plate

# Regex chuẩn cho tên file: [CamID]_[Index]_[RowType]_[PlateText]_[X1]_[Y1]_[X2]_[Y2].jpg
FILENAME_PATTERN = re.compile(r"^(\d+)_(\d+)_(\d+)_([A-Za-z0-9]+)_(\d+)_(\d+)_(\d+)_(\d+)\.jpg$")

raw_car_files = []
for folder in ["one_row", "two_rows"]:
    folder_path = SRC_ROOT / folder
    if folder_path.exists():
        with os.scandir(folder_path) as it:
            for entry in it:
                if entry.name.endswith(".jpg"):
                    raw_car_files.append((folder, entry.name, entry.path))

print(f"Tổng số ảnh ô tô quét được ban đầu: {len(raw_car_files)} ảnh")

Tổng số ảnh ô tô quét được ban đầu: 31704 ảnh


## 3. Phát hiện & Loại bỏ 25 Ảnh Chụp Dọc Dị Biệt (H > W)
* **Bản chất lý thuyết:** Trong 31.704 ảnh gốc, có đúng 25 ảnh người chụp quay dọc điện thoại/camera ($480 \times 640$). Tác giả bộ dữ liệu gốc khi gán nhãn đã bị tráo đổi trục tọa độ hoặc xoay ngang ảnh nhưng quên xoay nhãn, dẫn đến bounding box bị nằm ngang cắt ngang xe.
* **Giải pháp chuẩn:** Phát hiện tự động qua kích thước ($H > W$), in danh sách đối soát và loại bỏ hoàn toàn 25 ảnh này khỏi pipeline để dataset đạt độ tinh khiết 100%.

In [23]:
print("Đang kiểm tra kích thước ảnh để phát hiện ảnh chụp dọc dị biệt (H > W)...")
portrait_removed = []
car_files = []

def check_orientation(item):
    folder, name, path = item
    with Image.open(path) as im:
        w, h = im.size
    if h > w:
        return ("portrait", (name, w, h))
    else:
        return ("landscape", item)

with ThreadPoolExecutor(max_workers=16) as executor:
    results = list(executor.map(check_orientation, raw_car_files))

for r_type, data in results:
    if r_type == "portrait":
        portrait_removed.append(data)
    else:
        car_files.append(data)

print(f"-> Đã phát hiện và loại bỏ: {len(portrait_removed)} ảnh chụp dọc ngoại lai!")
print(f"-> Số lượng ảnh ô tô chuẩn ngang (W > H) còn lại: {len(car_files)} ảnh")

print("\nDanh sách 25 ảnh chụp dọc đã loại bỏ (kèm kích thước W x H):")
for idx, (p_name, pw, ph) in enumerate(portrait_removed, 1):
    print(f"{idx:2d}. {p_name} ({pw}x{ph})")

# Tự động xóa khỏi data/VNLP_detection nếu các file này từng lỡ được copy trước đó
cleaned_up = 0
for split in ["train", "val", "test"]:
    for p_name, _, _ in portrait_removed:
        img_target = DST_ROOT / "images" / split / p_name
        lbl_target = DST_ROOT / "labels" / split / (Path(p_name).stem + ".txt")
        if img_target.exists():
            img_target.unlink()
            cleaned_up += 1
        if lbl_target.exists():
            lbl_target.unlink()

if cleaned_up > 0:
    print(f"\nĐã dọn dẹp {cleaned_up} file ảnh/nhãn chụp dọc cũ trong thư mục {DST_ROOT}!")
else:
    print("\nThư mục đích data/VNLP_detection hoàn toàn sạch sẽ, không chứa ảnh dọc.")

Đang kiểm tra kích thước ảnh để phát hiện ảnh chụp dọc dị biệt (H > W)...
-> Đã phát hiện và loại bỏ: 25 ảnh chụp dọc ngoại lai!
-> Số lượng ảnh ô tô chuẩn ngang (W > H) còn lại: 31679 ảnh

Danh sách 25 ảnh chụp dọc đã loại bỏ (kèm kích thước W x H):
 1. 1_1976_1_15A08581_64_229_480_302.jpg (480x640)
 2. 1_2027_1_29A41952_71_208_480_290.jpg (480x640)
 3. 1_2051_1_29A94638_70_264_480_325.jpg (480x640)
 4. 1_2726_1_36A21316_143_235_480_293.jpg (480x640)
 5. 1_2729_1_36A21361_54_208_480_266.jpg (480x640)
 6. 1_2739_1_36A22201_61_206_480_280.jpg (480x640)
 7. 1_3302_1_36A38785_64_186_480_229.jpg (480x640)
 8. 2_1000_1_36A39246_25_233_480_323.jpg (480x640)
 9. 2_1005_1_36A39264_31_198_480_253.jpg (480x640)
10. 2_1021_1_36A39319_121_191_480_276.jpg (480x640)
11. 2_1502_1_36C27644_146_244_480_285.jpg (480x640)
12. 2_1648_1_36N6389_75_217_480_285.jpg (480x640)
13. 2_1672_1_61A20651_26_209_480_280.jpg (480x640)
14. 2_605_1_29A73407_20_213_480_261.jpg (480x640)
15. 2_791_1_36A11693_184_217_480_2

## 4. Gom nhóm theo Xe (Group by Car) & Phân chia Train / Val / Test (Zero-Leakage)
* Gom tất cả ảnh đầu xe và đuôi xe của cùng một xe vào chung một nhóm theo `clean_plate_id`.
* Chia nhóm xe theo tỷ lệ **80% Train : 10% Val : 10% Test** để đảm bảo **0% rò rỉ dữ liệu (0% Data Leakage)**.

In [ ]:
# Gom nhóm các ảnh thuộc cùng 1 xe
plate_groups = defaultdict(list)
for folder, name, path in car_files:
    m = FILENAME_PATTERN.match(name)
    if m:
        cam_id, idx, row_type, plate_raw, x1, y1, x2, y2 = m.groups()
        c_plate = clean_plate_id(plate_raw)
        plate_groups[c_plate].append({
            "name": name,
            "path": path,
            "coords": (int(x1), int(y1), int(x2), int(y2)),
            "folder": folder
        })

unique_plates = sorted(list(plate_groups.keys()))
random.seed(42)
random.shuffle(unique_plates)

n_total = len(unique_plates)
n_train = int(n_total * 0.8)
n_val = int(n_total * 0.1)

train_plates = set(unique_plates[:n_train])
val_plates = set(unique_plates[n_train:n_train + n_val])
test_plates = set(unique_plates[n_train + n_val:])

plate_to_split = {}
for p in train_plates: plate_to_split[p] = "train"
for p in val_plates: plate_to_split[p] = "val"
for p in test_plates: plate_to_split[p] = "test"

print(f"Tổng số xe (Unique Plates) : {n_total} xe")
print(f"- Xe tập Train             : {len(train_plates)} xe ({len(train_plates)/n_total*100:.1f}%)")
print(f"- Xe tập Val               : {len(val_plates)} xe ({len(val_plates)/n_total*100:.1f}%)")
print(f"- Xe tập Test              : {len(test_plates)} xe ({len(test_plates)/n_total*100:.1f}%)")

## 5. Thực thi Đa luồng: Copy Ảnh + Sinh Nhãn YOLO Chuẩn Hóa + Tạo data.yaml
* Đọc kích thước thật từng ảnh bằng PIL và tính toán tọa độ chuẩn hóa YOLO $[0, 1]$.
* Tự động sinh file cấu hình `data.yaml` với đường dẫn tương đối.

In [ ]:
# Tạo cấu trúc thư mục đích
for split in ["train", "val", "test"]:
    (DST_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (DST_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

# Chuẩn bị danh sách tác vụ
tasks = []
for plate, items in plate_groups.items():
    split = plate_to_split[plate]
    for it in items:
        tasks.append((it, split))

def process_task(task):
    it, split = task
    src_path = it["path"]
    name = it["name"]
    x1, y1, x2, y2 = it["coords"]
    
    dst_img = DST_ROOT / "images" / split / name
    dst_lbl = DST_ROOT / "labels" / split / (Path(name).stem + ".txt")
    
    # 1. Copy ảnh (chỉ copy nếu chưa có để tiết kiệm thời gian)
    if not dst_img.exists():
        shutil.copy2(src_path, dst_img)
        
    # 2. Đọc kích thước ảnh thực tế
    with Image.open(src_path) as im:
        W, H = im.size
        
    # 3. Tính toán tọa độ chuẩn hóa YOLO [0, 1]
    xc = ((x1 + x2) / 2.0) / float(W)
    yc = ((y1 + y2) / 2.0) / float(H)
    bw = (x2 - x1) / float(W)
    bh = (y2 - y1) / float(H)
    
    # 4. Ghi file nhãn .txt
    with open(dst_lbl, "w", encoding="utf-8") as f:
        f.write(f"0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")

print(f"Bắt đầu xử lý {len(tasks)} ảnh và nhãn bằng đa luồng...")
t0 = time.time()
with ThreadPoolExecutor(max_workers=16) as executor:
    list(executor.map(process_task, tasks))
print(f"Xử lý hoàn tất trong {time.time() - t0:.2f} giây!")

# Tạo file data.yaml
yaml_content = f"""path: .
train: images/train
val: images/val
test: images/test

names:
  0: license_plate
"""
yaml_file = DST_ROOT / "data.yaml"
with open(yaml_file, "w", encoding="utf-8") as f:
    f.write(yaml_content)
print(f"Đã tạo file cấu hình: {yaml_file}")

## 6. Kiểm tra Thống kê & Kiểm Tra Data Leakage
Kiểm tra lại số lượng ảnh, số lượng nhãn và xác minh độ rò rỉ ranh giới giữa các tập.

In [ ]:
print("=== BẢNG THỐNG KÊ DATASET SAU KHI XỬ LÝ ===")
split_plates = {}
for split in ["train", "val", "test"]:
    img_dir = DST_ROOT / "images" / split
    lbl_dir = DST_ROOT / "labels" / split
    
    imgs = os.listdir(img_dir)
    lbls = os.listdir(lbl_dir)
    
    plates = set(clean_plate_id(fn.split("_")[3]) for fn in imgs if FILENAME_PATTERN.match(fn))
    split_plates[split] = plates
    print(f"- Tập {split.upper():5s}: {len(imgs):6,d} ảnh | {len(lbls):6,d} nhãn | {len(plates):6,d} xe")

leak_train_val = len(split_plates["train"] & split_plates["val"])
leak_train_test = len(split_plates["train"] & split_plates["test"])
leak_val_test = len(split_plates["val"] & split_plates["test"])

print("\n=== KIỂM TRA RÒ RỈ DỮ LIỆU (DATA LEAKAGE) ===")
print(f"- Số xe trùng giữa Train & Val  : {leak_train_val} (Chuẩn tuyệt đối!)")
print(f"- Số xe trùng giữa Train & Test : {leak_train_test} (Chuẩn tuyệt đối!)")
print(f"- Số xe trùng giữa Val & Test   : {leak_val_test} (Chuẩn tuyệt đối!)")
print("-> ĐỘ RÒ RỈ RANH GIỚI: 0.00% (Hoàn hảo cho Machine Learning!)")

## 7. Trực quan hóa Bounding Box ngẫu nhiên để nghiệm thu
Vẽ thử Bounding Box từ nhãn `.txt` lên ảnh ngẫu nhiên để xác minh độ khớp.

In [ ]:
def visualize_samples(split="train", num_samples=6):
    img_dir = DST_ROOT / "images" / split
    lbl_dir = DST_ROOT / "labels" / split
    
    sample_files = random.sample(os.listdir(img_dir), num_samples)
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()
    
    for idx, fn in enumerate(sample_files):
        img_path = img_dir / fn
        lbl_path = lbl_dir / (Path(fn).stem + ".txt")
        
        im = Image.open(img_path)
        W, H = im.size
        draw = ImageDraw.Draw(im)
        
        with open(lbl_path, "r") as f:
            for line in f:
                cls, xc, yc, bw, bh = map(float, line.strip().split())
                x1 = (xc - bw / 2.0) * W
                y1 = (yc - bh / 2.0) * H
                x2 = (xc + bw / 2.0) * W
                y2 = (yc + bh / 2.0) * H
                draw.rectangle([x1, y1, x2, y2], outline="red", width=3)
                
        axes[idx].imshow(im)
        axes[idx].set_title(fn.split("_")[3], fontsize=10)
        axes[idx].axis("off")
        
    plt.tight_layout()
    plt.show()

visualize_samples(split="train", num_samples=6)